# 1장. 선형 모델에서 신경망으로

사람이 정하던 꺾이는 점을 모델이 자료에서 찾게 하면 무엇이 달라지는가.

**준비:** 한국전력거래소 「시간별 전국 전력수요량」 CSV 두 개(2023년, 2025년), 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개(2023년, 2025년), 국토교통부 서울 아파트 매매 실거래가 CSV 두 개(2023년, 2024년 계약분)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 3분쯤 걸린다.

## 1.1 사람이 정한 꺾이는 점으로 전력수요 예측하기

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

def bends(t, low=12, high=20):
    """low도보다 추운 만큼, high도보다 더운 만큼을 특성으로 만든다. 첫 열은 절편이다."""
    cold, hot = np.maximum(low - t, 0), np.maximum(t - high, 0)
    return np.column_stack([np.ones(len(t)), cold, hot])

def both(predict23, predict25):
    """2023년(훈련 자료)과 2025년(시험 자료)의 RMSE를 소수 둘째 자리까지."""
    return round(rmse(Y23, predict23), 2), round(rmse(Y25, predict25), 2)

line_x = np.column_stack([np.ones(len(T23)), T23])
straight = np.linalg.lstsq(line_x, Y23, rcond=None)[0]
by_hand = np.linalg.lstsq(bends(T23), Y23, rcond=None)[0]
print("평균만:", *both(np.full(len(Y23), Y23.mean()), np.full(len(Y25), Y23.mean())))
print("직선:", *both(straight[0] + straight[1] * T23, straight[0] + straight[1] * T25))
print("12도와 20도에서 꺾은 선:", *both(bends(T23) @ by_hand, bends(T25) @ by_hand))
print("계수:", by_hand.round(2).tolist())

In [ ]:
def bent_line(low, high):
    """꺾이는 점 두 개로 2023년 자료에서 계수를 구하고, 두 해의 RMSE를 돌려준다."""
    x23, x25 = bends(T23, low, high), bends(T25, low, high)
    coef = np.linalg.lstsq(x23, Y23, rcond=None)[0]
    return rmse(Y23, x23 @ coef), rmse(Y25, x25 @ coef)

pairs = [(12, 20), (10, 22), (5, 25), (15, 19)]
rows = [(low, high, *np.round(bent_line(low, high), 2)) for low, high in pairs]
names = ["낮은 쪽(°C)", "높은 쪽(°C)", "2023년 RMSE", "2025년 RMSE"]
print(pd.DataFrame(rows, columns=names).to_string(index=False))

# 0.5도 간격으로 두 점의 조합을 모두 넣어 보고 2023년 RMSE가 가장 작은 조합을 찾는다
lows, highs = np.arange(5, 20.1, 0.5), np.arange(10, 28.1, 0.5)
tried = [(bent_line(low, high)[0], low, high) for low in lows for high in highs if high >= low]
best = min(tried)
print(len(tried), round(best[0], 2), float(best[1]), float(best[2]))

## 1.2 유닛과 활성화 함수: 꺾이는 점을 모델이 찾게 하기

In [ ]:
def unit(x, w, b):
    """유닛 하나의 계산: 가중치를 곱하고 편향을 더한 뒤 0보다 작으면 0으로 바꾼다."""
    return np.maximum(0, w * x + b)

temps = np.array([4.0, 12.0, 16.0, 20.0, 30.0])        # 기온(°C) 다섯 개
print(unit(temps, -0.5, 6.0).tolist())                  # w = -0.5, b = 6
print(unit(temps, 0.5, -10.0).tolist())                 # w = 0.5, b = -10

In [ ]:
def network(x, w, b, v, c):
    """은닉층이 하나인 신경망. x: (N,)  w, b, v: (H,)  c: 숫자 하나  결과: (N,)"""
    z = np.outer(x, w) + b          # (N, H)  유닛마다 가중치를 곱하고 편향을 더한 값
    a = np.maximum(0, z)            # (N, H)  유닛의 출력
    return a @ v + c                # (N,)    유닛의 출력에 가중치를 곱해 더한 값

# 손으로 정한 값이라는 뜻으로 이름 끝에 _h를 붙였다
w_h, b_h = np.array([-0.5, 0.5]), np.array([6.0, -10.0])
v_h, c_h = np.array([2.2, 3.0]), 62.0
a_h = np.maximum(0, np.outer(temps, w_h) + b_h)
print(a_h.shape, (a_h @ v_h).shape)
print(np.round(network(temps, w_h, b_h, v_h, c_h), 1).tolist())
print(*both(network(T23, w_h, b_h, v_h, c_h), network(T25, w_h, b_h, v_h, c_h)))

In [ ]:
# 표준화에 쓰는 평균과 표준편차는 2023년에서만 구한다
t_mean, t_std = T23.mean(), T23.std()
y_mean, y_std = Y23.mean(), Y23.std()
xs, ys = (T23 - t_mean) / t_std, (Y23 - y_mean) / y_std
print(*np.round([t_mean, t_std, y_mean, y_std], 2).tolist())

def predict(theta, x):
    """theta = (w1, w2, b1, b2, v1, v2, c). 표준화한 기온 (N,)을 받아 예측 (N,)을 낸다."""
    return network(x, theta[0:2], theta[2:4], theta[4:6], theta[6])

def loss(theta):
    """2023년 자료에서 계산한 평균 제곱 오차. 표준화한 단위다."""
    return float(((predict(theta, xs) - ys) ** 2).mean())

def gradient(theta, eps=1e-5):
    """숫자를 하나씩 eps만큼 늘린 손실과 줄인 손실의 차이로 기울기를 구한다."""
    g = np.zeros(len(theta))
    for i in range(len(theta)):
        move = np.zeros(len(theta))
        move[i] = eps
        g[i] = (loss(theta + move) - loss(theta - move)) / (2 * eps)
    return g

def start(seed):
    """가중치 4개와 은닉층의 편향 2개는 표준정규분포에서 뽑고, 출력의 편향은 0이다."""
    return np.append(np.random.default_rng(seed).normal(0, 1, 6), 0.0)

theta0 = start(3)
print(theta0.round(2).tolist())
print(round(loss(theta0), 4), gradient(theta0).round(3).tolist())

In [ ]:
def learn(seed, steps=5000, rate=0.05):
    """경사하강법. 걸음마다 손실을 기록해서 마지막 theta와 함께 돌려준다."""
    theta, trace, kept = start(seed), [], {}
    for step in range(steps + 1):
        trace.append(loss(theta))
        if step in (0, 10, 100, 1000, steps):
            kept[step] = theta.copy()
        if step < steps:
            theta = theta - rate * gradient(theta)
    return theta, np.array(trace), kept

def gw(theta, t):
    """기온(°C)을 받아 전력수요(GW)를 낸다. 표준화, 예측, 원래 단위로 되돌리기 순서다."""
    return predict(theta, (t - t_mean) / t_std) * y_std + y_mean

def kinks(theta):
    """유닛마다 wx + b = 0이 되는 기온(°C)."""
    return -theta[2:4] / theta[0:2] * t_std + t_mean

theta3, trace3, kept3 = learn(3)
for step in (0, 10, 100, 1000, 5000):
    now = kept3[step]
    train_rmse = round(rmse(Y23, gw(now, T23)), 2)
    print(step, round(float(trace3[step]), 4), train_rmse, kinks(now).round(1).tolist())
print(theta3.round(2).tolist())
# 유닛이 켜진 구간에서 기온 1°C당 수요 변화(GW)와, 두 유닛이 모두 꺼진 구간의 수요(GW)
slopes = theta3[4:6] * theta3[0:2] * y_std / t_std
print(slopes.round(2).tolist(), round(float(theta3[6] * y_std + y_mean), 2))
print(*both(gw(theta3, T23), gw(theta3, T25)))

## 1.3 시드에 따른 학습 결과의 변동

In [ ]:
runs = [learn(seed)[0] for seed in range(10)]
rows = []
for seed, theta in enumerate(runs):
    # 유닛마다 2023년 247일 가운데 출력이 0보다 큰 날의 비율
    active = ((np.outer(xs, theta[0:2]) + theta[2:4]) > 0).mean(axis=0)
    steepest = float(np.abs(gradient(theta)).max())
    rows.append((seed, round(loss(theta), 4), *both(gw(theta, T23), gw(theta, T25)),
                 *kinks(theta).round(1).tolist(), *active.round(2).tolist(),
                 round(steepest, 4)))
names = ["시드", "손실", "23년", "25년", "점 1", "점 2", "켜짐 1", "켜짐 2", "기울기"]
table = pd.DataFrame(rows, columns=names)
print(table.to_string(index=False))
print(*np.round([T23.min(), T23.max()], 1).tolist(), int((table["25년"] < 3.5).sum()))

In [ ]:
swap = [1, 0, 3, 2, 5, 4, 6]                      # 유닛 1과 유닛 2의 번호를 바꾸는 순서
theta_a, theta_b = theta3, theta3[swap]
middle = (theta_a + theta_b) / 2
print(round(loss(theta_a), 4), round(loss(theta_b), 4), round(loss(middle), 4))
print(float(np.abs(predict(theta_a, xs) - predict(theta_b, xs)).max()))
print(middle.round(2).tolist())

## 1.4 유닛의 수와 층의 수 늘리기

In [ ]:
from sklearn.neural_network import MLPRegressor

def fit_net(units, seed, t=T23, y=Y23):
    """신경망을 학습시키고, 기온을 받아 수요(GW)를 내는 함수와 모델을 돌려준다."""
    net = MLPRegressor(hidden_layer_sizes=units, activation="relu", solver="lbfgs",
                       alpha=1e-4, max_iter=5000, random_state=seed)
    net.fit(((t - t_mean) / t_std)[:, None], (y - y_mean) / y_std)
    ask = lambda q: net.predict(((q - t_mean) / t_std)[:, None]) * y_std + y_mean
    return ask, net

def count(net):
    return sum(a.size for a in net.coefs_) + sum(a.size for a in net.intercepts_)

two = [fit_net((2,), seed) for seed in range(10)]
print(count(two[0][1]), sorted(round(rmse(Y25, f(T25)), 2) for f, _ in two))
best_train = min(rmse(Y23, f(T23)) for f, _ in two)
print(round(best_train, 2), round(rmse(Y23, gw(theta3, T23)), 2))

In [ ]:
rows, marks = [], {}
for units in [(2,), (4,), (8,), (32,), (128,), (8, 8), (32, 32)]:
    nets = [fit_net(units, seed) for seed in range(10)]
    m = np.array([[rmse(Y23, f(T23)), rmse(Y25, f(T25))] for f, _ in nets])
    marks[units] = m
    train, test = m[:, 0], m[:, 1]
    summary = [np.median(train), test.min(), np.median(test), test.max()]
    summary = np.round(summary, 2).tolist()
    rows.append((str(units), count(nets[0][1]), *summary, int((test > 4).sum())))
names = ["은닉층", "개수", "23년 중앙", "25년 최소", "25년 중앙", "25년 최대",
         "4GW 초과"]
print(pd.DataFrame(rows, columns=names).to_string(index=False))

In [ ]:
rng = np.random.default_rng(1)
rows = []
for days in [30, 60, 120, 247]:
    m = []
    for trial in range(10):
        # 2023년 평일 가운데 days일만 있다고 가정한다
        mine = rng.choice(len(Y23), days, replace=False)
        f, _ = fit_net((8,), trial, T23[mine], Y23[mine])
        coef = np.linalg.lstsq(bends(T23[mine]), Y23[mine], rcond=None)[0]
        m.append((rmse(Y25, f(T25)), rmse(Y25, bends(T25) @ coef)))
    m = np.array(m)
    rows.append((days, *m.mean(axis=0).round(2), *m.max(axis=0).round(2)))
names = ["훈련 일수", "신경망 평균", "꺾은 선 평균", "신경망 최대", "꺾은 선 최대"]
print(pd.DataFrame(rows, columns=names).to_string(index=False))

## 1.5 표 자료에 그대로 적용한 신경망과 1권의 모델 비교

In [ ]:
import os, re

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트를 돌려준다."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)

v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["동이름"] = v["구"] + " " + v["동네"]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["연"] = v["날짜"].dt.year
v["연식"] = v["연"] - v["건축년도"]
v["달"] = (v["연"] - 2023) * 12 + v["날짜"].dt.month          # 2023년 1월이 1, 2025년 12월이 36
v = v.sort_values("날짜").reset_index(drop=True)
print(len(files), raw.shape, len(v))

In [ ]:
from scipy import sparse
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

def tables(ref):
    """ref의 동과 단지에 번호를 매기고, 트리용 표와 릿지용 표를 만드는 함수를 돌려준다."""
    dongs = pd.Index(sorted(ref["동이름"].unique()))
    complexes = pd.Index(sorted(ref["단지"].unique()))
    def for_trees(df):
        dong_no = dongs.get_indexer(df["동이름"])
        complex_no = complexes.get_indexer(df["단지"])
        plain = [df["면적"], df["층"], df["건축년도"]]
        return np.column_stack([*plain, dong_no, complex_no, df["달"]])
    def for_line(df):
        age = (2023 + (df["달"] - 1) // 12) - df["건축년도"]
        decades = [(np.clip(age, 0, 59) // 10 == k).astype(float) for k in range(1, 6)]
        num = np.column_stack([np.log(df["면적"]), df["층"], *decades, df["달"]])
        blocks = [sparse.csr_matrix(num)]
        for col, cats in [("동이름", dongs), ("단지", complexes)]:
            where = cats.get_indexer(df[col])
            known = where >= 0
            ones = (np.ones(known.sum()), (np.where(known)[0], where[known]))
            blocks.append(sparse.csr_matrix(ones, shape=(len(df), len(cats))))
        return sparse.hstack(blocks).tocsr()
    return for_trees, for_line

def price(df):
    return np.log(df["억"].to_numpy())

def fit(made, kind, lam=0.01, rounds=1000, seed=0):
    """made로 모델을 추정하고, 거래 표를 받아 로그 가격을 예측하는 함수를 돌려준다."""
    for_trees, for_line = tables(made)
    line = Ridge(alpha=lam, solver="sparse_cg", tol=1e-8, max_iter=20000)
    line.fit(for_line(made), price(made))
    if kind == "릿지":
        return lambda df: line.predict(for_line(df))
    mend = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=rounds,
                                         early_stopping=False, random_state=seed)
    # 릿지의 잔차를 부스팅이 예측한다
    mend.fit(for_trees(made), price(made) - line.predict(for_line(made)))
    return lambda df: line.predict(for_line(df)) + mend.predict(for_trees(df))

def off(actual_log, predicted_log):
    """오차율의 중앙값(%)."""
    ratio = np.abs(np.exp(predicted_log - actual_log) - 1)
    return round(float(np.median(ratio) * 100), 2)

In [ ]:
a23, a24 = v[v["연"] == 2023], v[v["연"] == 2024]
for kind in ["릿지", "릿지+부스팅"]:
    model = fit(a23, kind)
    print(kind, off(price(a23), model(a23)), off(price(a24), model(a24)))

for_trees, _ = tables(a23)
# 열의 순서: 면적, 층, 건축년도, 동 번호, 단지 번호, 달
raw23, raw24 = for_trees(a23), for_trees(a24)
unseen = raw24[:, 4] < 0            # 2023년에 거래가 없던 단지는 번호가 -1이다
print(raw23.shape, a23["동이름"].nunique(), a23["단지"].nunique())
print(int(unseen.sum()), round(float(unseen.mean() * 100), 1))

def tab_net(cols, seed):
    """cols에 적은 열만 표준화해서 넣는다. 두 해의 오차율과 에폭 수를 돌려준다."""
    center, scale = raw23[:, cols].mean(axis=0), raw23[:, cols].std(axis=0)
    in23, in24 = (raw23[:, cols] - center) / scale, (raw24[:, cols] - center) / scale
    net = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=500, random_state=seed)
    net.fit(in23, price(a23))
    train, test = off(price(a23), net.predict(in23)), off(price(a24), net.predict(in24))
    return train, test, net.n_iter_

apt_marks = []
for seed in range(3):
    mark = tab_net([0, 1, 2, 3, 4, 5], seed)
    apt_marks.append(mark[1])
    print("신경망 시드", seed, *mark)

In [ ]:
choices = [("달을 뺀 다섯 열", [0, 1, 2, 3, 4]),
           ("단지 번호를 뺀 다섯 열", [0, 1, 2, 3, 5]),
           ("동 번호와 단지 번호를 뺀 네 열", [0, 1, 2, 5]),
           ("면적, 층, 건축년도 세 열", [0, 1, 2])]
for name, cols in choices:
    got = [tab_net(cols, seed) for seed in range(3)]
    print(name, [g[0] for g in got], [g[1] for g in got])

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 1-1
grid_t = np.linspace(-10, 31, 411)
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.axvline(12, color=GRAY, lw=0.8, ls=(0, (2, 2)))
ax.axvline(20, color=GRAY, lw=0.8, ls=(0, (2, 2)))
ax.scatter(T23, Y23, s=11, color=to_rgba(GRAY, 0.75), edgecolor="none", label="2023년 평일 247일")
ax.plot(grid_t, straight[0] + straight[1] * grid_t, color=NAVY, lw=1.5, ls=(0, (5, 2)),
        label=f"직선: 2025년 RMSE {rmse(Y25, straight[0] + straight[1] * T25):.2f}GW")
ax.plot(grid_t, bends(grid_t) @ by_hand, color=BRICK, lw=2.2,
        label=f"12°C와 20°C에서 꺾은 선: 2025년 RMSE {rmse(Y25, bends(T25) @ by_hand):.2f}GW")
ax.text(11.6, 83.2, "12°C", fontsize=9, color="#5b6470", ha="right")
ax.text(20.4, 83.2, "20°C", fontsize=9, color="#5b6470", ha="left")
ax.text(16, 53.4, f"12°C와 20°C 사이의 높이 {by_hand[0]:.2f}GW", fontsize=9, color=BRICK, ha="center")
ax.text(-9.4, 59.0, f"1°C 낮아질 때\n{by_hand[1]:.2f}GW 증가", fontsize=9, color=BRICK, ha="left", va="center", linespacing=1.5)
ax.text(30.6, 57.6, f"1°C 높아질 때\n{by_hand[2]:.2f}GW 증가", fontsize=9, color=BRICK, ha="right", va="center", linespacing=1.5)
ax.set_xlim(-10.5, 31.5); ax.set_ylim(52, 94); ax.set_xticks(range(-10, 31, 5))
ax.set_xlabel("일평균 기온(°C)", fontsize=9)
ax.set_ylabel("일평균 전력수요(GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_1.png", dpi=300)
plt.show()

In [ ]:
# 그림 1-4
xs_plot = np.linspace(-6, 36, 421)
fig, (top, bot) = plt.subplots(2, 1, figsize=(7, 7.4), gridspec_kw={"height_ratios": [1, 1.25], "hspace": 0.42})
for ax in (top, bot):
    ax.axvline(12, color=GRAY, lw=0.8, ls=(0, (2, 2)))
    ax.axvline(20, color=GRAY, lw=0.8, ls=(0, (2, 2)))
    ax.set_xlim(-6, 36); ax.set_xticks(range(-5, 36, 5))
    ax.set_xlabel("입력 x: 기온(°C)", fontsize=9)
    ax.tick_params(labelbottom=True)
top.plot(xs_plot, unit(xs_plot, w_h[0], b_h[0]), color=NAVY, lw=1.9, label="유닛 1의 출력 $a_1=\\max(0,\\ -0.5x+6)$")
top.plot(xs_plot, unit(xs_plot, w_h[1], b_h[1]), color=BRICK, lw=1.9, ls=(0, (5, 2)), label="유닛 2의 출력 $a_2=\\max(0,\\ 0.5x-10)$")
top.scatter(temps, unit(temps, w_h[0], b_h[0]), s=26, color=NAVY, zorder=5, edgecolor="white", lw=0.7)
top.scatter(temps, unit(temps, w_h[1], b_h[1]), s=26, color=BRICK, zorder=6, edgecolor="white", lw=0.7, marker="s")
top.text(11.4, 10.9, "유닛 1이 꺾이는 점 12°C\n$-b_1/w_1=-6/(-0.5)$", fontsize=9, color=NAVY, ha="right", va="top", linespacing=1.6)
top.text(20.6, 10.9, "유닛 2가 꺾이는 점 20°C\n$-b_2/w_2=10/0.5$", fontsize=9, color=BRICK, ha="left", va="top", linespacing=1.6)
top.text(4.9, 4.9, "x = 4일 때 4.0", fontsize=9, color=NAVY, ha="left", va="center")
top.text(29.1, 5.9, "x = 30일 때 5.0", fontsize=9, color=BRICK, ha="right", va="center")
top.set_ylim(-0.6, 11.5); top.set_yticks(range(0, 11, 2))
top.set_ylabel("유닛의 출력", fontsize=9)
top.legend(frameon=False, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, 1.09), ncol=2, columnspacing=2.2)
frame(top, "유닛 두 개의 출력: 꺾이는 점의 한쪽에서만 0보다 크다")
out5 = network(temps, w_h, b_h, v_h, c_h)
bot.plot(xs_plot, network(xs_plot, w_h, b_h, v_h, c_h), color=TEAL, lw=2.4)
bot.scatter(temps, out5, s=34, color=TEAL, zorder=5, edgecolor="white", lw=0.8)
for tx, ty, dx, dy in zip(temps, out5, [1.9, 0, 0, 0, -1.9], [1.5, 2.3, 2.3, 2.3, 1.5]):
    bot.text(tx + dx, ty + dy, f"{ty:.1f}", fontsize=9, color=TEAL, ha="center", va="center", weight="bold")
bot.text(-5.2, 86.6, "x < 12: 기울기 $v_1w_1=2.2\\times(-0.5)=-1.1$", fontsize=9, color=NAVY, ha="left", va="center")
bot.text(35.7, 86.6, "x > 20: 기울기 $v_2w_2=3.0\\times 0.5=1.5$", fontsize=9, color=BRICK, ha="right", va="center", bbox=BOX)
bot.text(16, 58.4, "12 ≤ x ≤ 20: 두 유닛의 출력이 모두 0\n신경망의 출력은 c = 62", fontsize=9, color="#5b6470", ha="center", va="center", linespacing=1.6, bbox=BOX)
bot.set_ylim(55, 90); bot.set_yticks(range(55, 91, 5))
bot.set_ylabel("신경망의 출력(GW)", fontsize=9)
frame(bot, "신경망의 출력 $c+v_1a_1+v_2a_2$: 두 점에서 꺾이는 선", color=TEAL)
fig.savefig("fig_1_4.png", dpi=300, bbox_inches="tight", pad_inches=0.12)
plt.show()

In [ ]:
# 그림 1-6
fig, (top, bot) = plt.subplots(2, 1, figsize=(7, 7.6), gridspec_kw={"height_ratios": [1, 1.3], "hspace": 0.4})
steps = np.arange(1, 5001)
top.plot(steps, trace3[1:], color=BRICK, lw=1.9)
for step, dx, dy, ha in [(10, 1.25, 0.1, "left"), (100, 1.0, 0.2, "center"), (1000, 1.0, 0.44, "center"), (5000, 1.25, 0.2, "right")]:
    top.scatter([step], [trace3[step]], s=30, color=BRICK, zorder=5, edgecolor="white", lw=0.8)
    top.text(step * dx, trace3[step] + dy, f"{step:,}걸음: {trace3[step]:.4f}", fontsize=9, color=BRICK, ha=ha, va="center")
top.text(1.05, 0.34, f"시작할 때의 손실 {trace3[0]:.4f}", fontsize=9, color="#5b6470", ha="left", va="center")
top.set_xscale("log"); top.set_xlim(0.9, 6500); top.set_ylim(0, 2.4)
plain_ticks(top.xaxis)
top.set_xlabel("경사하강법의 걸음 수(로그 축)", fontsize=9)
top.set_ylabel("손실(표준화한 단위)", fontsize=9)
top.tick_params(labelbottom=True)
frame(top, "손실: 걸음이 늘어날수록 줄어들고 1,000걸음 뒤에는 거의 변하지 않는다", color=BRICK)
bot.scatter(T23, Y23, s=10, color=to_rgba(GRAY, 0.6), edgecolor="none")
styles = {0: (GRAY, (0, (2, 2)), 1.3), 10: (NAVY, (0, (5, 2)), 1.3), 100: (NAVY, "-", 1.4), 5000: (TEAL, "-", 2.4)}
for step, (col, ls, lw) in styles.items():
    bot.plot(grid_t, gw(kept3[step], grid_t), color=col, ls=ls, lw=lw,
             label=("시작" if step == 0 else f"{step:,}걸음") + f": 2023년 RMSE {rmse(Y23, gw(kept3[step], T23)):.2f}GW")
bot.set_xlim(-10.5, 31.5); bot.set_ylim(54, 94); bot.set_xticks(range(-10, 31, 5))
bot.set_xlabel("일평균 기온(°C)", fontsize=9)
bot.set_ylabel("일평균 전력수요(GW)", fontsize=9)
bot.legend(frameon=False, fontsize=9, loc="upper center", ncol=2)
frame(bot, "예측: 시작할 때의 선에서 자료의 모양으로 바뀌어 간다", color=TEAL)
fig.savefig("fig_1_6.png", dpi=300, bbox_inches="tight", pad_inches=0.12)
plt.show()

In [ ]:
# 그림 1-8
from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(7, 5.0))
ax.scatter(T23, Y23, s=10, color=to_rgba(GRAY, 0.6), edgecolor="none")
test = table["25년"].to_numpy()
for seed, theta in enumerate(runs):
    good = test[seed] < 3.5
    ax.plot(grid_t, gw(theta, grid_t), color=TEAL if good else BRICK, lw=2.0 if good else 1.4, ls="-" if good else (0, (5, 2)), zorder=4 if good else 3)
notes = [(84.6, "시드 0, 3, 5, 9", TEAL), (79.2, "시드 8: 6.21GW", BRICK), (71.0, "시드 4: 6.81GW", BRICK), (68.3, "시드 2, 7: 6.58GW", BRICK), (65.7, "시드 1, 6: 5.20GW", BRICK)]
for y, text, col in notes:
    ax.text(-11.0, y, text, fontsize=9, color=col, ha="right", va="center")
ax.text(-23.4, 56.0, "선 왼쪽의 숫자는\n2025년 RMSE", fontsize=9, color="#5b6470", ha="left", va="center", linespacing=1.5)
ax.set_xlim(-24, 31.5); ax.set_ylim(54, 94); ax.set_xticks(range(-10, 31, 5))
ax.set_xlabel("일평균 기온(°C)", fontsize=9)
ax.set_ylabel("일평균 전력수요(GW)", fontsize=9)
handles = [Line2D([], [], color=TEAL, lw=2.0, label="2025년 RMSE 3.5GW 미만: 시드 " + ", ".join(str(k) for k in np.where(test < 3.5)[0])),
           Line2D([], [], color=BRICK, lw=1.4, ls=(0, (5, 2)), label="3.5GW 이상: 시드 " + ", ".join(str(k) for k in np.where(test >= 3.5)[0]))]
ax.legend(handles=handles, frameon=False, fontsize=9, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_8.png", dpi=300)
plt.show()

In [ ]:
# 그림 1-9
alphas = np.linspace(-0.25, 1.25, 301)
along = np.array([loss((1 - a) * theta_a + a * theta_b) for a in alphas])
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(alphas, along, color=BRICK, lw=2.0, label="해 A와 해 B(해 A에서 유닛 번호만 바꾼 것)를 잇는 선분 위의 손실")
ax.plot([0, 1], [loss(theta_a), loss(theta_b)], color=NAVY, lw=1.3, ls=(0, (5, 2)), label="볼록 함수라면 손실이 이 선보다 위에 있을 수 없다")
ax.scatter([0, 1], [loss(theta_a), loss(theta_b)], s=40, color=TEAL, zorder=5, edgecolor="white", lw=0.8)
ax.scatter([0.5], [loss(middle)], s=40, color=BRICK, zorder=5, edgecolor="white", lw=0.8)
ax.text(0, 0.42, f"해 A\n손실 {loss(theta_a):.4f}", fontsize=9, color=TEAL, ha="center", va="bottom", linespacing=1.5)
ax.text(1, 0.42, f"해 B\n손실 {loss(theta_b):.4f}", fontsize=9, color=TEAL, ha="center", va="bottom", linespacing=1.5)
ax.text(0.5, loss(middle) + 0.12, f"두 해의 중간\n손실 {loss(middle):.4f}", fontsize=9, color=BRICK, ha="center", va="bottom", linespacing=1.5)
ax.set_xlim(-0.27, 1.27); ax.set_ylim(0, 3.2)
ax.set_xticks([-0.25, 0, 0.25, 0.5, 0.75, 1, 1.25])
ax.set_xlabel("선분 위의 위치 α (0이면 해 A, 1이면 해 B)", fontsize=9)
ax.set_ylabel("손실(표준화한 단위)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_9.png", dpi=300)
plt.show()

In [ ]:
# 그림 1-10
names = list(marks)
fig, ax = plt.subplots(figsize=(7, 4.8))
for k, units in enumerate(names):
    ax.scatter(np.full(10, k), marks[units][:, 1], s=16, color=to_rgba(GRAY, 0.8), edgecolor="none", zorder=3)
ax.scatter([], [], s=16, color=to_rgba(GRAY, 0.8), label="2025년 RMSE: 시드 10개 각각")
ax.plot(range(len(names)), [np.median(marks[u][:, 1]) for u in names], color=TEAL, lw=2.0, marker="o", ms=5, label="2025년 RMSE: 시드 10개의 중앙값", zorder=5)
ax.plot(range(len(names)), [np.median(marks[u][:, 0]) for u in names], color=NAVY, lw=1.4, ls=(0, (5, 2)), marker="s", ms=4, label="2023년 RMSE(훈련 자료): 중앙값", zorder=4)
for k, u in enumerate(names):
    mid = np.median(marks[u][:, 1])
    ax.text(k + 0.07, mid + 0.1, f"{mid:.2f}", fontsize=9, color=TEAL, ha="left", va="bottom", weight="bold")
hinge25 = rmse(Y25, bends(T25) @ by_hand)
ax.axhline(hinge25, color=BRICK, lw=1.2, ls=(0, (1.5, 1.5)), label=f"12°C와 20°C에서 꺾은 선: 2025년 {hinge25:.2f}GW")
ax.set_xticks(range(len(names)))
ax.set_xticklabels([f"{u}\n{n}개" for u, n in zip(names, [7, 13, 25, 97, 385, 97, 1153])], fontsize=9)
ax.set_xlim(-0.5, len(names) - 0.5); ax.set_ylim(2.2, 7.2)
ax.set_xlabel("은닉층의 유닛 수와, 가중치와 편향의 수", fontsize=9)
ax.set_ylabel("RMSE(GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_10.png", dpi=300)
plt.show()

In [ ]:
# 그림 1-11
labels = ["릿지 + 부스팅", "릿지"] + [f"신경망(시드 {k})" for k in range(3)]
values = [off(price(a24), fit(a23, "릿지+부스팅")(a24)), off(price(a24), fit(a23, "릿지")(a24))] + apt_marks
colors = [TEAL, NAVY, BRICK, BRICK, BRICK]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(range(len(values))[::-1], values, color=[to_rgba(c, 0.85) for c in colors], height=0.6)
for k, val in enumerate(values):
    ax.text(val + 0.4, len(values) - 1 - k, f"{val:.2f}%", fontsize=9, color=colors[k], va="center", weight="bold")
ax.set_yticks(range(len(values))[::-1]); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlim(0, max(values) + 4)
ax.set_xlabel("2024년 거래에서 계산한 오차율의 중앙값(%)", fontsize=9)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_1_11.png", dpi=300)
plt.show()